In [1]:
from typing import Any

import torch
import torch.nn as nn
import numpy as np

In [2]:
class DGMValueNet(nn.Module):
    """
    Position-only value function V(q, t, gpos):
      q: (7,)
      t: scalar normalized to [0,1]
      gpos: (3,) goal position in planning frame
    Input dim = 7 + 1 + 3 = 11
    Output = scalar V
    """

    def __init__(self, in_dim=16, hidden=256, depth=4):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.Tanh()]
            d = hidden
        layers += [nn.Linear(d, in_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        print(f"input dgm shape {x.shape}")
        return self.net(x).squeeze(-1)

In [3]:
# class DGMLayer_(nn.Module):
#     """
#     Georgias Detorakis (2024): Practical Aspects on Solving Differential Equations Using Deep Learning: A Primer

#     """

#     def __init__(self, input_dim=1, hidden_size=50):
#         super().__init__()

#         self.I_zu = nn.Linear(input_dim, hidden_size)
#         self.Z_wg = nn.Linear(hidden_size, hidden_size)
#         self.Z_ug = nn.Linear(input_dim, hidden_size, bias=False)

#         self.G_wz = nn.Linear(hidden_size, hidden_size)
#         self.G_uz = nn.Linear(input_dim, hidden_size, bias=False)

#         self.R_wr = nn.Linear(hidden_size, hidden_size)
#         self.R_ur = nn.Linear(input_dim, hidden_size, bias=False)

#         self.H_wh = nn.Linear(hidden_size, hidden_size)
#         self.H_uh = nn.Linear(input_dim, hidden_size, bias=False)

#         # Non−linear Activation function
#         self.sigma = nn.Tanh()

#     def forward(self, x, s):
#         I = self.I_zu(s)
#         print(f"I shape = {I.shape}")
#         Z = self.sigma(self.Z_wg(I) + self.Z_ug(x))
#         print(f"Z shape = {Z.shape}")
#         G = self.sigma(self.G_wz(Z) + self.G_uz(x))
#         print(f"G shape = {G.shape}")
#         R = self.sigma(self.R_wr(G) + self.R_ur(x))
#         print(f"R shape {R.shape} and s shape {s.shape} and I shape {self.H_wh(I).shape}")
#         H = self.sigma(self.H_wh(I) * R + self.H_uh(x))
#         print(f"H shape {H.shape}")
#         out = (1-G)*H + Z*self.H_wh(I)
#         print(f"out shape {out.shape}")
#         # out = torch.sub(1, G) * H + Z * s
#         return out



In [4]:
class DGMLayer(nn.Module):
    """
    Georgias Detorakis (2024): Practical Aspects on Solving Differential Equations Using Deep Learning: A Primer

    """

    def __init__(self, input_dim=1, hidden_size=50):
        super().__init__()

        self.Z_wg = nn.Linear(hidden_size, hidden_size)
        self.Z_ug = nn.Linear(input_dim, hidden_size, bias=False)

        self.G_wz = nn.Linear(hidden_size, hidden_size)
        self.G_uz = nn.Linear(input_dim, hidden_size, bias=False)

        self.R_wr = nn.Linear(hidden_size, hidden_size)
        self.R_ur = nn.Linear(input_dim, hidden_size, bias=False)

        self.H_wh = nn.Linear(hidden_size, hidden_size)
        self.H_uh = nn.Linear(input_dim, hidden_size, bias=False)

        # Non−linear Activation function
        self.sigma = nn.Tanh()

    def forward(self, x, s):
#         I = self.I_zu(s)
#         print(f"I shape = {I.shape}")
        Z = self.sigma(self.Z_wg(s) + self.Z_ug(x))
        print(f"Z shape = {Z.shape}")
        G = self.sigma(self.G_wz(Z) + self.G_uz(x))
        print(f"G shape = {G.shape}")
        R = self.sigma(self.R_wr(G) + self.R_ur(x))
        print(f"R shape {R.shape} and s shape {s.shape} and I shape {self.H_wh(s).shape}")
        H = self.sigma(self.H_wh(s) * R + self.H_uh(x))
        print(f"H shape {H.shape}")
        out = (1-G)*H + Z*self.H_wh(s)
        print(f"out shape {out.shape}")
        # out = torch.sub(1, G) * H + Z * s
        return out



In [5]:
class DGMLayer0(nn.Module):

    def __init__(self, input_dim=1, hidden_size=50):
        super().__init__()

        self.I_zu = nn.Linear(input_dim, hidden_size)
        self.dgm_layer = DGMLayer(input_dim, hidden_size)
    

    def forward(self, x, s):
        s1 = self.I_zu(s)
        print(f"s1 shape {s1.shape}")
        out = self.dgm_layer(x,s1)
        print(f"out shape {out.shape}")
        return out



In [6]:
class DGMLayerN(nn.Module):

    def __init__(self, input_dim=1, output_dim=1, hidden_size=50):
        super().__init__()

        self.dgm_layer = DGMLayer(input_dim, hidden_size)
        self.K_zu = nn.Linear(hidden_size, output_dim)
    

    def forward(self, x, s):
        
        x1 = self.dgm_layer(x,s)
        print(f"x1 shape {x1.shape}")
        out = self.K_zu(x1)
        print(f"out shape {out.shape}")
        return out

In [7]:
def build_input(q, t_norm, gpos):
    # q: (B,7), t_norm: (B,1), gpos: (B,3)
    return torch.cat([q, t_norm, gpos], dim=-1)

In [8]:
def sample_goals(n):
    # Sample from xyz boundaries (cuboid) from which
    # to train the NN to generate plans to reach points inside this goals cuboid/region
    xs = np.random.uniform(0.25, 0.65, (n, 1))
    ys = np.random.uniform(-0.30, 0.30, (n, 1))
    zs = np.random.uniform(0.10, 0.60, (n, 1))
    return np.hstack([xs, ys, zs]).astype(np.float64)

In [9]:
jmin = np.array([-2.8973, -1.7628, -2.8973, -3.0718, -2.8973, -0.0175, -2.8973], dtype=np.float64)
jmax = np.array([2.8973, 1.7628, 2.8973, -0.0698, 2.8973, 3.7525, 2.8973], dtype=np.float64)
batch = 192
T = 2
Qp = 10

In [10]:
q_np = np.random.uniform(jmin, jmax, (batch, 7)).astype(np.float64)
t_np = np.random.uniform(0.0, T, (batch, 1)).astype(np.float64)
g_np = sample_goals(batch)

In [11]:
g_np.shape

(192, 3)

In [12]:


# running cost via FK (position-only)
# l_np = position_loss_fn(fk, joint_names, batch, Qp, g_np, q_np)

l_np = np.zeros((batch,), dtype=np.float64)
for i in range(batch):
    try:
#         p = fk.ee_position(joint_names, q_np[i])  # fk client gets coordinate position of hand/end-effector
        p = np.random.rand(3) 
        e = p - g_np[i]  # distance between current joint position i and goal position i
        l_np[i] = Qp * float(np.dot(e, e))
    except Exception:
        rospy.logwarn("fk_pos l: couldn't retrieve fk position")
        l_np[i] = 1e3


In [13]:
device = torch.device('cpu')

In [14]:
l_np.shape

(192,)

In [15]:
q = torch.tensor(q_np, dtype=torch.float32, device=device, requires_grad=True)
t = torch.tensor((t_np / T), dtype=torch.float32, device=device, requires_grad=True)
g = torch.tensor(g_np, dtype=torch.float32, device=device)
l = torch.tensor(l_np, dtype=torch.float32, device=device)

In [16]:
print(q.shape)
print(t.shape)
print(g.shape)
print(l.shape)

torch.Size([192, 7])
torch.Size([192, 1])
torch.Size([192, 3])
torch.Size([192])


In [17]:
inp = build_input(q, t, g)

In [18]:
inp.shape

torch.Size([192, 11])

In [19]:
inp.T.shape

torch.Size([11, 192])

In [20]:
dgm_layer = DGMLayer(input_dim=11, hidden_size=192)

In [21]:
dgm_layer

DGMLayer(
  (Z_wg): Linear(in_features=192, out_features=192, bias=True)
  (Z_ug): Linear(in_features=11, out_features=192, bias=False)
  (G_wz): Linear(in_features=192, out_features=192, bias=True)
  (G_uz): Linear(in_features=11, out_features=192, bias=False)
  (R_wr): Linear(in_features=192, out_features=192, bias=True)
  (R_ur): Linear(in_features=11, out_features=192, bias=False)
  (H_wh): Linear(in_features=192, out_features=192, bias=True)
  (H_uh): Linear(in_features=11, out_features=192, bias=False)
  (sigma): Tanh()
)

In [22]:
dgm_layer_0 = DGMLayer0(input_dim=11, hidden_size=192)

In [23]:
dgm_layer_0

DGMLayer0(
  (I_zu): Linear(in_features=11, out_features=192, bias=True)
  (dgm_layer): DGMLayer(
    (Z_wg): Linear(in_features=192, out_features=192, bias=True)
    (Z_ug): Linear(in_features=11, out_features=192, bias=False)
    (G_wz): Linear(in_features=192, out_features=192, bias=True)
    (G_uz): Linear(in_features=11, out_features=192, bias=False)
    (R_wr): Linear(in_features=192, out_features=192, bias=True)
    (R_ur): Linear(in_features=11, out_features=192, bias=False)
    (H_wh): Linear(in_features=192, out_features=192, bias=True)
    (H_uh): Linear(in_features=11, out_features=192, bias=False)
    (sigma): Tanh()
  )
)

In [24]:
init = dgm_layer_0(inp,inp)

s1 shape torch.Size([192, 192])
Z shape = torch.Size([192, 192])
G shape = torch.Size([192, 192])
R shape torch.Size([192, 192]) and s shape torch.Size([192, 192]) and I shape torch.Size([192, 192])
H shape torch.Size([192, 192])
out shape torch.Size([192, 192])
out shape torch.Size([192, 192])


In [25]:
# V = model(build_input(q, t, g))
# loss_pde = hjb_residual_loss(V, q, t, l,
#                              R_inv_diag)  # hjb_residual_loss(V, q, t_norm, running_cost, R_inv_diag)



In [26]:
dgm_layer_n = DGMLayerN(input_dim=11,output_dim=192, hidden_size=192)

In [27]:
dgm_layer_n(inp, init)

Z shape = torch.Size([192, 192])
G shape = torch.Size([192, 192])
R shape torch.Size([192, 192]) and s shape torch.Size([192, 192]) and I shape torch.Size([192, 192])
H shape torch.Size([192, 192])
out shape torch.Size([192, 192])
x1 shape torch.Size([192, 192])
out shape torch.Size([192, 192])


tensor([[-0.1580, -0.3417, -0.0438,  ...,  0.4998,  0.2653,  0.3019],
        [-0.6904,  0.2290,  0.1663,  ...,  0.0238,  0.0182, -0.3468],
        [-0.1563,  0.1182, -0.1302,  ..., -0.2563, -0.2794, -0.2911],
        ...,
        [-1.2441,  0.4805, -0.1569,  ..., -0.0141, -0.2762, -0.1943],
        [-0.6837,  0.2341, -0.1695,  ..., -0.0704, -0.0795, -0.1078],
        [-0.5503,  0.1996, -0.0614,  ..., -0.3732, -0.4419, -0.1443]],
       grad_fn=<AddmmBackward0>)

In [28]:
class ValueNet(nn.Module):
    """
    num_dgm_layers 
    """

    def __init__(self, num_layers=1, input_dim=1, output_dim=1, hidden_size=50):
        super().__init__()
        
        self.layers = nn.ModuleList([DGMLayer0(input_dim, hidden_size)]) + \
        nn.ModuleList([DGMLayer(input_dim,hidden_size) for _ in range(num_layers)]) + \
                      nn.ModuleList([DGMLayerN(input_dim, output_dim, hidden_size)])   
            
#         self.dgm_layer = DGMLayer(input_dim, hidden_size)
    

    def forward(self, x, s):
        
        for i, layer in enumerate(self.layers):
            print(f"layer {i} = {layer}")
            x = layer(s,x)

        print(f"out shape: {x.shape}")
        return x.squeeze(-1)

In [29]:
v_net = ValueNet(num_layers=2, input_dim=11, output_dim=1, hidden_size=192)

In [30]:
v_net

ValueNet(
  (layers): ModuleList(
    (0): DGMLayer0(
      (I_zu): Linear(in_features=11, out_features=192, bias=True)
      (dgm_layer): DGMLayer(
        (Z_wg): Linear(in_features=192, out_features=192, bias=True)
        (Z_ug): Linear(in_features=11, out_features=192, bias=False)
        (G_wz): Linear(in_features=192, out_features=192, bias=True)
        (G_uz): Linear(in_features=11, out_features=192, bias=False)
        (R_wr): Linear(in_features=192, out_features=192, bias=True)
        (R_ur): Linear(in_features=11, out_features=192, bias=False)
        (H_wh): Linear(in_features=192, out_features=192, bias=True)
        (H_uh): Linear(in_features=11, out_features=192, bias=False)
        (sigma): Tanh()
      )
    )
    (1-2): 2 x DGMLayer(
      (Z_wg): Linear(in_features=192, out_features=192, bias=True)
      (Z_ug): Linear(in_features=11, out_features=192, bias=False)
      (G_wz): Linear(in_features=192, out_features=192, bias=True)
      (G_uz): Linear(in_features=11

In [31]:
v = v_net(inp,inp)

layer 0 = DGMLayer0(
  (I_zu): Linear(in_features=11, out_features=192, bias=True)
  (dgm_layer): DGMLayer(
    (Z_wg): Linear(in_features=192, out_features=192, bias=True)
    (Z_ug): Linear(in_features=11, out_features=192, bias=False)
    (G_wz): Linear(in_features=192, out_features=192, bias=True)
    (G_uz): Linear(in_features=11, out_features=192, bias=False)
    (R_wr): Linear(in_features=192, out_features=192, bias=True)
    (R_ur): Linear(in_features=11, out_features=192, bias=False)
    (H_wh): Linear(in_features=192, out_features=192, bias=True)
    (H_uh): Linear(in_features=11, out_features=192, bias=False)
    (sigma): Tanh()
  )
)
s1 shape torch.Size([192, 192])
Z shape = torch.Size([192, 192])
G shape = torch.Size([192, 192])
R shape torch.Size([192, 192]) and s shape torch.Size([192, 192]) and I shape torch.Size([192, 192])
H shape torch.Size([192, 192])
out shape torch.Size([192, 192])
out shape torch.Size([192, 192])
layer 1 = DGMLayer(
  (Z_wg): Linear(in_features=1

In [32]:
v.squeeze(-1).shape

torch.Size([192])

In [33]:
class DGMLayer_(nn.Module):
    """
    Georgias Detorakis (2024): Practical Aspects on Solving Differential Equations Using Deep Learning: A Primer

    """

    def __init__(self, input_dim=1, hidden_size=50, expansion_factor=2):
        super().__init__()
        
        self.expanded_hidden_size = expansion_factor*hidden_size

        self.Z_wg = nn.Linear(self.expanded_hidden_size, self.expanded_hidden_size)
        self.Z_ug = nn.Linear(input_dim, self.expanded_hidden_size, bias=False)

        self.G_wz = nn.Linear(expansion_factor*hidden_size, expansion_factor*hidden_size)
        self.G_uz = nn.Linear(input_dim, self.expanded_hidden_size, bias=False)

        self.R_wr = nn.Linear(self.expanded_hidden_size, self.expanded_hidden_size)
        self.R_ur = nn.Linear(input_dim, self.expanded_hidden_size, bias=False)

        self.H_wh = nn.Linear(self.expanded_hidden_size, self.expanded_hidden_size)
        self.H_uh = nn.Linear(input_dim, self.expanded_hidden_size, bias=False)

        # Non−linear Activation function
        self.sigma = nn.Tanh()

    def forward(self, x, s):
        
        Z = self.sigma(self.Z_wg(s) + self.Z_ug(x))
        print(f"Z shape = {Z.shape}")
        G = self.sigma(self.G_wz(Z) + self.G_uz(x))
        print(f"G shape = {G.shape}")
        R = self.sigma(self.R_wr(G) + self.R_ur(x))
        print(f"R shape {R.shape} and s shape {s.shape} and I shape {self.H_wh(s).shape}")
        H = self.sigma(self.H_wh(s) * R + self.H_uh(x))
        print(f"H shape {H.shape}")
        out = (1-G)*H + Z*self.H_wh(s)
        print(f"out shape {out.shape}")
        # out = torch.sub(1, G) * H + Z * s
        return out



In [34]:
class DGMLayer0_(nn.Module):

    def __init__(self, input_dim=1, hidden_size=50, expansion_factor=2):
        super().__init__()

        self.I_zu = nn.Linear(input_dim, expansion_factor*hidden_size)
        self.dgm_layer = DGMLayer_(input_dim, hidden_size, expansion_factor=2)
    

    def forward(self, x, s):
        s1 = self.I_zu(s)
        print(f"s1 shape {s1.shape}")
        out = self.dgm_layer(x,s1)
        print(f"out shape {out.shape}")
        return out

In [35]:
dgm_layer0_ = DGMLayer0_(input_dim=11, hidden_size=192)

In [36]:
dgm_layer0_

DGMLayer0_(
  (I_zu): Linear(in_features=11, out_features=384, bias=True)
  (dgm_layer): DGMLayer_(
    (Z_wg): Linear(in_features=384, out_features=384, bias=True)
    (Z_ug): Linear(in_features=11, out_features=384, bias=False)
    (G_wz): Linear(in_features=384, out_features=384, bias=True)
    (G_uz): Linear(in_features=11, out_features=384, bias=False)
    (R_wr): Linear(in_features=384, out_features=384, bias=True)
    (R_ur): Linear(in_features=11, out_features=384, bias=False)
    (H_wh): Linear(in_features=384, out_features=384, bias=True)
    (H_uh): Linear(in_features=11, out_features=384, bias=False)
    (sigma): Tanh()
  )
)

In [37]:
y = dgm_layer0_(inp,inp)
y.shape

s1 shape torch.Size([192, 384])
Z shape = torch.Size([192, 384])
G shape = torch.Size([192, 384])
R shape torch.Size([192, 384]) and s shape torch.Size([192, 384]) and I shape torch.Size([192, 384])
H shape torch.Size([192, 384])
out shape torch.Size([192, 384])
out shape torch.Size([192, 384])


torch.Size([192, 384])

In [38]:
dgm_layer_ = DGMLayer_(input_dim=11, hidden_size=192, expansion_factor=2)

In [39]:
z = dgm_layer_(inp, y)

Z shape = torch.Size([192, 384])
G shape = torch.Size([192, 384])
R shape torch.Size([192, 384]) and s shape torch.Size([192, 384]) and I shape torch.Size([192, 384])
H shape torch.Size([192, 384])
out shape torch.Size([192, 384])


In [40]:
class DGMLayerN_(nn.Module):

    def __init__(self, input_dim=1, output_dim=1, hidden_size=192, expansion_factor=2):
        super().__init__()
        
        self.expanded_hidden_size = expansion_factor*hidden_size

        self.dgm_layer = DGMLayer_(input_dim, hidden_size, expansion_factor)
        self.dgm_layerN_ = nn.Linear(expansion_factor*hidden_size, hidden_size)
        self.K_zu = nn.Linear(hidden_size, output_dim)
    

    def forward(self, x, s):
        
        x1 = self.dgm_layer(x,s)
        print(f"x1 shape {x1.shape}")
        x2 = self.dgm_layerN_(x1)
        print(f"x2 shape {x2.shape}")
        out = self.K_zu(x2)
        print(f"out shape {out.shape}")
        return out

In [41]:
dgm_layerN_ = DGMLayerN_(input_dim=11, output_dim=1, hidden_size=192, expansion_factor=2)

In [42]:
dgm_layerN_

DGMLayerN_(
  (dgm_layer): DGMLayer_(
    (Z_wg): Linear(in_features=384, out_features=384, bias=True)
    (Z_ug): Linear(in_features=11, out_features=384, bias=False)
    (G_wz): Linear(in_features=384, out_features=384, bias=True)
    (G_uz): Linear(in_features=11, out_features=384, bias=False)
    (R_wr): Linear(in_features=384, out_features=384, bias=True)
    (R_ur): Linear(in_features=11, out_features=384, bias=False)
    (H_wh): Linear(in_features=384, out_features=384, bias=True)
    (H_uh): Linear(in_features=11, out_features=384, bias=False)
    (sigma): Tanh()
  )
  (dgm_layerN_): Linear(in_features=384, out_features=192, bias=True)
  (K_zu): Linear(in_features=192, out_features=1, bias=True)
)

In [43]:
w = dgm_layerN_(inp, z)
w.shape

Z shape = torch.Size([192, 384])
G shape = torch.Size([192, 384])
R shape torch.Size([192, 384]) and s shape torch.Size([192, 384]) and I shape torch.Size([192, 384])
H shape torch.Size([192, 384])
out shape torch.Size([192, 384])
x1 shape torch.Size([192, 384])
x2 shape torch.Size([192, 192])
out shape torch.Size([192, 1])


torch.Size([192, 1])

In [44]:


class ValueNet_(nn.Module):
    """
    num_dgm_layers 
    """

    def __init__(self, num_layers=1, input_dim=1, output_dim=1, hidden_size=50, expansion_factor=2):
        super().__init__()
        
        self.dropout = nn.Dropout(p=0.1)
        
        self.layers = nn.ModuleList([DGMLayer0_(input_dim, hidden_size, expansion_factor)]) + nn.ModuleList([DGMLayer_(input_dim, hidden_size, expansion_factor) for _ in range(num_layers)]) + nn.ModuleList([DGMLayerN_(input_dim, output_dim, hidden_size, expansion_factor)])   
            
#         self.dgm_layer = DGMLayer(input_dim, hidden_size)
    

    def forward(self, x, s):
        
        for i, layer in enumerate(self.layers):
            print(f"layer {i} = {layer}")
            x = layer(s,x)

        print(f"out shape: {x.shape}")
        return x.squeeze(-1)

In [45]:
v_net_ = ValueNet_(num_layers=2, input_dim=11, output_dim=1, hidden_size=192, expansion_factor=2)

In [46]:
v_net_

ValueNet_(
  (dropout): Dropout(p=0.1, inplace=False)
  (layers): ModuleList(
    (0): DGMLayer0_(
      (I_zu): Linear(in_features=11, out_features=384, bias=True)
      (dgm_layer): DGMLayer_(
        (Z_wg): Linear(in_features=384, out_features=384, bias=True)
        (Z_ug): Linear(in_features=11, out_features=384, bias=False)
        (G_wz): Linear(in_features=384, out_features=384, bias=True)
        (G_uz): Linear(in_features=11, out_features=384, bias=False)
        (R_wr): Linear(in_features=384, out_features=384, bias=True)
        (R_ur): Linear(in_features=11, out_features=384, bias=False)
        (H_wh): Linear(in_features=384, out_features=384, bias=True)
        (H_uh): Linear(in_features=11, out_features=384, bias=False)
        (sigma): Tanh()
      )
    )
    (1-2): 2 x DGMLayer_(
      (Z_wg): Linear(in_features=384, out_features=384, bias=True)
      (Z_ug): Linear(in_features=11, out_features=384, bias=False)
      (G_wz): Linear(in_features=384, out_features=384,

In [47]:
v0 = v_net_(inp, inp)
v0.shape

layer 0 = DGMLayer0_(
  (I_zu): Linear(in_features=11, out_features=384, bias=True)
  (dgm_layer): DGMLayer_(
    (Z_wg): Linear(in_features=384, out_features=384, bias=True)
    (Z_ug): Linear(in_features=11, out_features=384, bias=False)
    (G_wz): Linear(in_features=384, out_features=384, bias=True)
    (G_uz): Linear(in_features=11, out_features=384, bias=False)
    (R_wr): Linear(in_features=384, out_features=384, bias=True)
    (R_ur): Linear(in_features=11, out_features=384, bias=False)
    (H_wh): Linear(in_features=384, out_features=384, bias=True)
    (H_uh): Linear(in_features=11, out_features=384, bias=False)
    (sigma): Tanh()
  )
)
s1 shape torch.Size([192, 384])
Z shape = torch.Size([192, 384])
G shape = torch.Size([192, 384])
R shape torch.Size([192, 384]) and s shape torch.Size([192, 384]) and I shape torch.Size([192, 384])
H shape torch.Size([192, 384])
out shape torch.Size([192, 384])
out shape torch.Size([192, 384])
layer 1 = DGMLayer_(
  (Z_wg): Linear(in_feature

torch.Size([192])

In [48]:
import torch
import torch.nn as nn

class DGMLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(DGMLayer, self).__init__()
        # Standard fully connected layers used for the internal gates
        self.Uz = nn.Linear(input_dim, hidden_dim)
        self.Ug = nn.Linear(input_dim, hidden_dim)
        self.Ur = nn.Linear(input_dim, hidden_dim)
        self.Uh = nn.Linear(input_dim, hidden_dim)
        
        self.Wz = nn.Linear(hidden_dim, hidden_dim)
        self.Wg = nn.Linear(hidden_dim, hidden_dim)
        self.Wr = nn.Linear(hidden_dim, hidden_dim)
        self.Wh = nn.Linear(hidden_dim, hidden_dim)

        self.activation = nn.Tanh()

    def forward(self, x, S):
        # x is the input coordinates (spatial/temporal)
        # S is the output of the previous layer
        z = self.activation(self.Uz(x) + self.Wz(S))
        g = self.activation(self.Ug(x) + self.Wg(S))
        r = self.activation(self.Ur(x) + self.Wr(S))
        h = self.activation(self.Uh(x) + self.Wh(S * r))
        
        # Element-wise gate update
        return (1 - g) * h + z * S


In [49]:
class DGMNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim=1):
        super(DGMNet, self).__init__()
        self.initial_layer = nn.Linear(input_dim, hidden_dim)
        self.dgm_layers = nn.ModuleList([
            DGMLayer(input_dim, hidden_dim) for _ in range(num_layers)
        ])
        self.final_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Initial transformation
        S = torch.tanh(self.initial_layer(x))
        
        # Pass through specialized DGM layers
        for layer in self.dgm_layers:
            S = layer(x, S)
            
        return self.final_layer(S)


In [50]:
dgm_net = DGMNet(11, 192, 4)

In [51]:
dgm_net(inp).shape

torch.Size([192, 1])

In [52]:
import torch
import torch.nn as nn

class ResBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.dgm = DGMValueNet(in_dim=48, hidden=192, depth=4)
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        
        # Shortcut to align dimensions for the skip connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        x = self.dgm(x)
        print(f"dgm {x.shape}")
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return self.relu(out)

class ResNet1D(nn.Module):
    def __init__(self, input_channels=11, out_channels=64, num_layers=16, num_classes=10, dim=16):
        super().__init__()
        # Initial stem: reduces temporal length from 192 -> 48
        self.prep = nn.Sequential(
            nn.Conv1d(input_channels, out_channels, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )
        self.ln = nn.Conv1d(11,256,kernel_size=1, stride=1, padding=1, bias=False)
        
        self.layers = nn.ModuleList([ResBlock1D(out_channels, out_channels) for _ in range(num_layers)])
        
        # Residual Layers
        self.layer1 = ResBlock1D(out_channels, out_channels)
        self.layer2 = ResBlock1D(out_channels, 128, stride=2) # Length 48 -> 24
        
        # Output head
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, num_classes)
        
    def forward(self, x):
        print(x.shape)
        a, b = x.shape
        
        y = nn.Conv1d(a,192,kernel_size=3, stride=1, padding=1, bias=False)(x)
        print(f"shape after ln {y.shape}")
        c, d = y.shape
        x = y.T.reshape([1, d, c])
        
        x = self.prep(x)
        print(f"after prep {x.shape}")
        
        for i, layer in enumerate(self.layers):
            x = layer(x)
            print(f"x loop {x.shape}")
            
        print(f"after resblocks {x.shape}")
        x = self.layer2(x)
        print(f"layer2 {x.shape}")
        x = self.avgpool(x).squeeze(-1)
        print(f"x avg pool {x.shape}")
        x = nn.Linear(128,a)(x)
        print(f"x lin {x.shape}")
#         x = self.fc(x)
        return x

In [53]:
res_inp = inp.T.reshape([1,11, 192])
res_inp.shape

torch.Size([1, 11, 192])

In [54]:
# Verification
model = ResNet1D(input_channels=11, out_channels=192, num_layers=16, num_classes=192)
dummy_data = torch.randn(64, 11) # (Batch, Channels, Length)
# output = model(res_inp)
# print(f"Input: {dummy_data.shape} -> Output: {output.shape}")

In [55]:
dummy_data.shape

torch.Size([64, 11])

In [56]:
d = DGMValueNet(in_dim=48, hidden=192, depth=4)

In [57]:
# d(dummy_data)

In [58]:
model

ResNet1D(
  (prep): Sequential(
    (0): Conv1d(11, 192, kernel_size=(7,), stride=(2,), padding=(3,), bias=False)
    (1): BatchNorm1d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool1d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (ln): Conv1d(11, 256, kernel_size=(1,), stride=(1,), padding=(1,), bias=False)
  (layers): ModuleList(
    (0-15): 16 x ResBlock1D(
      (dgm): DGMValueNet(
        (net): Sequential(
          (0): Linear(in_features=48, out_features=192, bias=True)
          (1): Tanh()
          (2): Linear(in_features=192, out_features=192, bias=True)
          (3): Tanh()
          (4): Linear(in_features=192, out_features=192, bias=True)
          (5): Tanh()
          (6): Linear(in_features=192, out_features=192, bias=True)
          (7): Tanh()
          (8): Linear(in_features=192, out_features=48, bias=True)
        )
      )
      (conv1): Conv1d(192, 192, kernel_size=(3,),

In [59]:
output = model(dummy_data)

torch.Size([64, 11])
shape after ln torch.Size([192, 11])
after prep torch.Size([1, 192, 48])
input dgm shape torch.Size([1, 192, 48])
dgm torch.Size([1, 192, 48])
x loop torch.Size([1, 192, 48])
input dgm shape torch.Size([1, 192, 48])
dgm torch.Size([1, 192, 48])
x loop torch.Size([1, 192, 48])
input dgm shape torch.Size([1, 192, 48])
dgm torch.Size([1, 192, 48])
x loop torch.Size([1, 192, 48])
input dgm shape torch.Size([1, 192, 48])
dgm torch.Size([1, 192, 48])
x loop torch.Size([1, 192, 48])
input dgm shape torch.Size([1, 192, 48])
dgm torch.Size([1, 192, 48])
x loop torch.Size([1, 192, 48])
input dgm shape torch.Size([1, 192, 48])
dgm torch.Size([1, 192, 48])
x loop torch.Size([1, 192, 48])
input dgm shape torch.Size([1, 192, 48])
dgm torch.Size([1, 192, 48])
x loop torch.Size([1, 192, 48])
input dgm shape torch.Size([1, 192, 48])
dgm torch.Size([1, 192, 48])
x loop torch.Size([1, 192, 48])
input dgm shape torch.Size([1, 192, 48])
dgm torch.Size([1, 192, 48])
x loop torch.Size([1

In [60]:
output.shape

torch.Size([1, 64])

In [61]:
64/48


1.3333333333333333

In [90]:
T=2.0
batch=10

In [91]:
x = np.random.uniform(T, T*1.3, (batch, 1))

In [140]:
x.shape

(10, 1)

In [93]:
xs = np.sort(x.flatten()).reshape((batch,1))

In [94]:
xs

array([[2.04511483],
       [2.1189445 ],
       [2.13909261],
       [2.22207335],
       [2.32259658],
       [2.38604835],
       [2.41369664],
       [2.44385666],
       [2.45971156],
       [2.58600962]])

In [109]:
xs.shape

(10, 1)

In [149]:
result = np.tile(xs, (2,1))
result.shape

(20, 1)

In [155]:
np.repeat([xs], 3, axis=0).shape

(3, 10, 1)

In [139]:
result

array([[[2.04511483],
        [2.1189445 ],
        [2.13909261],
        [2.22207335],
        [2.32259658],
        [2.38604835],
        [2.41369664],
        [2.44385666],
        [2.45971156],
        [2.58600962]],

       [[2.04511483],
        [2.1189445 ],
        [2.13909261],
        [2.22207335],
        [2.32259658],
        [2.38604835],
        [2.41369664],
        [2.44385666],
        [2.45971156],
        [2.58600962]]])

In [102]:
ls=[]
i=0
for i in range(0,len(x)-1):
    diff=xs[i+1]-xs[i]
    ls.append()
    print(f"i, diff: {i} {xs[i+1]} {xs[i]}--> {diff}")
    i+=1

i, diff: 0 [2.1189445] [2.04511483]--> [0.07382967]
i, diff: 1 [2.13909261] [2.1189445]--> [0.02014811]
i, diff: 2 [2.22207335] [2.13909261]--> [0.08298074]
i, diff: 3 [2.32259658] [2.22207335]--> [0.10052323]
i, diff: 4 [2.38604835] [2.32259658]--> [0.06345177]
i, diff: 5 [2.41369664] [2.38604835]--> [0.02764829]
i, diff: 6 [2.44385666] [2.41369664]--> [0.03016002]
i, diff: 7 [2.45971156] [2.44385666]--> [0.0158549]
i, diff: 8 [2.58600962] [2.45971156]--> [0.12629806]


In [160]:
a = torch.zeros([8,7])
b = torch.zeros([82,7])
c = torch.zeros([192,7])

In [163]:
c = torch.cat([a,b,c])
c.shape

torch.Size([372, 7])